# AIGC ?????????????

? Notebook ??????????????????? CSV/PNG ???????????????????????????????????????? clean ???????????????????????????

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'report').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLE_DIR = PROJECT_ROOT / 'report' / 'tables'
FIG_DIR = PROJECT_ROOT / 'report' / 'figures'

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


def show_csv(name, n=10, columns=None):
    path = TABLE_DIR / name
    print(path)
    df = pd.read_csv(path)
    if columns:
        df = df[columns]
    display(df.head(n))
    return df


def show_fig(name, width=900):
    path = FIG_DIR / name
    print(path)
    display(Image(filename=str(path), width=width))

## 1. ????

?? GenImage ???????ADM?BigGAN?VQDM?GLIDE????? `train` split????? `val` split???????? AI/nature???????? AI ?????????

In [ ]:
dataset_counts = show_csv('dataset_counts.csv')

## 2. ?? clean ???

???????? `outputs_v2_full_best`?`fusion_freq + flat LightGBM + wide profile + full training`???????? stable feature ?????????????????

In [ ]:
v2 = show_csv('optimization_v2_summary.csv', n=12, columns=[
    'run', 'feature_profile', 'model_architecture', 'train_augmentation',
    'sample_fraction', 'binary_ai_vs_nature_macro_f1',
    'ai_subsource_attribution_macro_f1', 'combined_macro_f1', 'selected'
])

In [ ]:
show_fig('optimization_v2_macro_f1.png')
show_fig('scaleup_macro_f1.png')

## 3. ???? vs ????

???????? color / multiscale / block-DCT / residual ????? profile ? fusion profile?????????????? profile??????? 5% ? full ??????

In [ ]:
show_csv('single_vs_fusion_profiles.csv', n=20, columns=[
    'profile', 'family', 'task', 'sample_fraction', 'macro_f1', 'accuracy', 'run'
])
show_fig('single_vs_fusion_feature_profiles.png')
show_fig('feature_ablation_binary_ai_vs_nature.png')
show_fig('feature_ablation_ai_subsource_attribution.png')

## 4. ? generator ??

Leave-one-generator-out ???????????????????????????????? held-out generator???????????????? AIGC ???????????????

In [ ]:
logo = show_csv('logo_generalization.csv', n=10, columns=[
    'heldout_generator', 'train_generators', 'sample_fraction',
    'feature_profile', 'accuracy', 'macro_f1', 'auc'
])
show_fig('logo_generalization_macro_f1.png')

## 5. ?????

?????????????????? validation ??????? JPEG ????????????????????????????????????? degraded ?????????

In [ ]:
robust_cmp = show_csv('robustness_comparison.csv', n=25, columns=[
    'run', 'task', 'attack', 'level', 'n_samples', 'macro_f1', 'accuracy', 'auc'
])

In [ ]:
show_fig('robustness_comparison_binary_ai_vs_nature.png')
show_fig('robustness_comparison_ai_subsource_attribution.png')

## 6. Clean-vs-Robustness ??

`fusion_freq` clean ?????? degraded ?? F1 ????`stable_freq` clean ??????? degraded ???????`robust_aug` ??? degraded ?? F1 ????? clean ??????????

In [ ]:
tradeoff = show_csv('robustness_tradeoff.csv', n=20)
show_fig('robustness_tradeoff_binary_ai_vs_nature.png')
show_fig('robustness_tradeoff_ai_subsource_attribution.png')

## 11. ?????????

?????????????????? ADM/VQDM ???????????????????????????????????

In [ ]:
show_fig('confusion_binary_ai_vs_nature_lightgbm.png', width=650)
show_fig('confusion_ai_subsource_attribution_lightgbm.png', width=650)
show_csv('top_features_binary_ai_vs_nature.csv', n=20)

## 11. ????????

???????margin ? entropy ?????? coverage-accuracy ????????????????????????????????????????????????

In [ ]:
show_csv('confidence_by_generator_binary_ai_vs_nature.csv')
show_csv('confidence_by_generator_ai_subsource_attribution.csv')
show_fig('confidence_coverage_binary_ai_vs_nature.png')
show_fig('confidence_histogram_binary_ai_vs_nature.png')

## 11. ??????

????? generator????????????????? ADM/VQDM ? AI ???????? nature???? ADM ? VQDM ???????????

In [ ]:
show_csv('v2_error_breakdown_binary_ai_vs_nature.csv', n=10)
show_csv('v2_error_breakdown_ai_subsource_attribution.csv', n=10)

## 10. ????

???????????????????????????????????????????????????????????

In [ ]:
print(r'''
# final clean route
.\.venv\Scripts\python.exe test.py `
  --dataset-root C:\Users\99303\git\GenImage_data `
  --out-dir outputs_v2_full_best `
  --sample-fraction 1.0 --sample-seed 42 `
  --feature-profile fusion_freq --feature-set all `
  --lgbm-profile wide --model-set lightgbm --model-architecture flat `
  --train-augmentation none --calibrate-threshold `
  --lightgbm-device gpu --num-workers 16 --feature-chunksize 64 `
  --feature-cache-dir feature_cache_fusion --sample-cache-dir sample_cache `
  --skip-robustness --resume-completed-tasks

# robustness diagnostic
.\.venv\Scripts\python.exe scripts\evaluate_best_robustness.py `
  --dataset-root C:\Users\99303\git\GenImage_data `
  --model-output outputs_v2_full_best `
  --out-dir outputs_v2_full_best_robust_20pct `
  --sample-fraction 0.20 --sample-seed 42 --tasks both `
  --num-workers 16 --feature-chunksize 64 `
  --robust-cache-dir robustness_cache_fusion

# extension suite
.\scripts\run_all_4gen_extension_experiments.ps1 `
  -DatasetRoot C:\Users\99303\git\GenImage_data `
  -LightgbmDevice cpu -LogoFraction 0.20 `
  -CandidateFraction 0.20 -RobustnessFraction 0.20

# final validation
.\.venv\Scripts\python.exe scripts\validate_submission.py `
  --dataset-root C:\Users\99303\git\GenImage_data --write-report
''')

## 11. ??????

???????????? clean AIGC ???????????? degraded evaluation ? leave-one-generator-out ????????????????? JPEG?resize?noise ? unseen generator ??????????????????????????????????????????